# Рекомендатель казахских книг

Модель на эмбеддингах: аннотация каждой книги превращается в вектор, «похожие книги» =
ближайшие по косинусу. Плюс поиск по интересам на **любом языке** (для иностранцев).

**Файлы рядом:** `books.csv` (данные), `rec_vectors.npy`/`rec_names.npy` (векторы), `app.py` (сайт).

Порядок: запусти ячейки сверху вниз.

In [3]:
# зависимости (если не стоят) — раскомментируй
# %pip install -q sentence-transformers numpy

import os, csv
import numpy as np
from sentence_transformers import SentenceTransformer

# многоязычная модель: казахский, русский, английский в ОДНОМ пространстве
MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
print("ok")

ok


## 1) Данные из books.csv

In [4]:
def load_books(path="books.csv"):
    "читает книги из csv; возвращает (names, descriptions)"
    with open(path, encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    return [r["name"] for r in rows], [r["description"] for r in rows]

names, descriptions = load_books()
print("загружено книг:", len(names))
print("пример:", names[0], "->", descriptions[0][:50], "...")

загружено книг: 37
пример: Абай жолы — Мұхтар Әуезов -> Абайдың өмірі мен ақындық жолы туралы эпопея, XIX  ...


## 2) Аннотации -> векторы

In [5]:
model = SentenceTransformer(MODEL)

def embed(texts):
    "тексты -> нормализованные векторы (единичные, для косинуса)"
    return model.encode(list(texts), convert_to_numpy=True, normalize_embeddings=True)

item_vectors = embed(descriptions)   # (n_books, 384)
print("готово:", item_vectors.shape[0], "книг ->", item_vectors.shape[1], "dims")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

готово: 37 книг -> 384 dims


## 3) Похожие на книгу

In [6]:
def top_k(sims, k, skip=None):
    "индексы k наибольших sims, пропуская skip"
    order = np.argsort(-sims)
    return [j for j in order if j != skip][:k]

def recommend(name, topn=3):
    "печатает topn похожих на name книг"
    i = names.index(name)
    sims = item_vectors @ item_vectors[i]     # косинус ко всем
    print(f"похожие на «{name}»:")
    for j in top_k(sims, topn, skip=i):
        print(f"  {sims[j]:.3f}  {names[j]}")

recommend("Көшпенділер — Ілияс Есенберлин")
print()
recommend("Бақытсыз Жамал — Міржақып Дулатов")

похожие на «Көшпенділер — Ілияс Есенберлин»:
  0.985  Абай жолы — Мұхтар Әуезов
  0.969  Қазақ солдаты — Ғабит Мүсірепов
  0.960  Қалың мал — Спандияр Көбеев

похожие на «Бақытсыз Жамал — Міржақып Дулатов»:
  0.871  Ақбілек — Жүсіпбек Аймауытов
  0.845  Ұлпан — Ғабит Мүсірепов
  0.834  Еңлік-Кебек — Мұхтар Әуезов


## 4) Поиск по интересам (любой язык)

Иностранец пишет запрос на английском/русском — многоязычная модель находит казахские книги.

In [7]:
def search_by_interest(query, topn=3):
    "по тексту-запросу (любой язык) находит подходящие книги"
    qv = embed([query])[0]
    sims = item_vectors @ qv
    print(f'запрос: "{query}"')
    for j in top_k(sims, topn):
        print(f"  {sims[j]:.3f}  {names[j]}")

search_by_interest("a funny story about childhood and school")   # English
print()
search_by_interest("трагическая судьба женщины")                 # Russian
print()
search_by_interest("тарих, хандар мен батырлар")                 # Kazakh

запрос: "a funny story about childhood and school"
  0.471  Қорғансыздың күні — Мұхтар Әуезов
  0.445  Менің атым Қожа — Бердібек Соқпақбаев
  0.421  Ақ боз үй — Смағұл Елубай

запрос: "трагическая судьба женщины"
  0.648  Бақытсыз Жамал — Міржақып Дулатов
  0.506  Қозы Көрпеш – Баян сұлу — халық жыры
  0.467  Ұлпан — Ғабит Мүсірепов

запрос: "тарих, хандар мен батырлар"
  0.776  Қыз Жібек — халық жыры
  0.772  Шақан-Шері — Мұхтар Мағауин
  0.762  Қан мен тер — Әбдіжәміл Нұрпейісов


## 5) Сохранить векторы для app.py

In [8]:
np.save("rec_vectors.npy", item_vectors)
np.save("rec_names.npy", np.array(names, dtype=object))
print("сохранено: rec_vectors.npy, rec_names.npy")
print("запуск сайта:  streamlit run app.py")

сохранено: rec_vectors.npy, rec_names.npy
запуск сайта:  streamlit run app.py
